# 7. Pose blend shapes

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

A rigid skeleton can only rotate the skin. On a real hand, bending a joint bulges the skin and creases it. MANO approximates that with a linear correction called a pose blend shape.

## Where the 135 numbers come from

Skip the wrist. Fifteen joints remain. Each joint becomes a 3×3 rotation `R`, then `R - I`. `R - I` has 9 numbers, read in row-major order. 15 × 9 = 135.

```text
pose_feature = concat( R_1 - I, R_2 - I, ..., R_15 - I )     # length 135
v_posed      = v_shaped + posedirs · pose_feature
```

On the official model `posedirs` is `(778, 3, 135)`. It turns those 135 numbers into a 3D displacement at every vertex.

The wrist is left out on purpose. If it were included, spinning the whole hand in the air would change `R_0 - I` and would change the flesh correction. With the root removed, the pose blend responds only to how the fingers bend relative to the palm.

When every finger rotation is the identity, every block of `R - I` is 0 and the correction is 0. The flat template does not grow a bump from nothing.

## Ninety degrees on the toy

The toy has 3 non-root joints, so its feature length is 3 × 9 = 27. That plays the same role as 135.

Rotate only the PIP by 90 degrees about X. Chapter 2 already produced

```text
R - I = [[0, 0, 0],
         [0,-1,-1],
         [0, 1,-1]]
```

The toy `posedirs` places 0.05 on the z component of the skin vertex, in the matching channel. Before skinning, the skin vertex moves from `(0, 2, 0.3)` to `(0, 2, 0.35)`. Rotating the wrist by another 30 degrees about X should leave those 27 numbers unchanged.


In [1]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

import toy_mano as toy

pose = np.zeros((4, 3))
pose[2, 0] = np.deg2rad(90.0)
spun = pose.copy()
spun[0, 0] = np.deg2rad(30.0)

feature = toy.pose_feature(pose)
print("feature length", feature.size, "norm", round(float(np.linalg.norm(feature)), 4))
print("feature unchanged after a 30-degree wrist rotation?", np.allclose(feature, toy.pose_feature(spun)))

with_blend = toy.forward(pose)
rigid = toy.forward(pose, use_pose_blend=False)
gap = with_blend["v_posed"][4] - rigid["v_posed"][4]
print("correction of the skin vertex before skinning", np.round(gap, 4))
print("v_posed after the correction", np.round(with_blend["v_posed"][4], 4))

assert feature.size == 27
assert np.allclose(feature, toy.pose_feature(spun))
assert np.isclose(gap[2], 0.05)
assert np.allclose(toy.pose_feature(np.zeros(12)), 0.0)


feature length 27 norm 2.0
feature unchanged after a 30-degree wrist rotation? True
correction of the skin vertex before skinning [0.   0.   0.05]
v_posed after the correction [0.   2.   0.35]


## What it is, and where it fails

The correction is linear in `R - I`. It is fit from poses seen during training, to make up the difference between "rigid bones plus skinning" and real scans. It is not a muscle, and it has no separate collision. Far from the training poses, for example with a finger folded past any real angle, the linear correction produces a surface that no longer looks like a hand.

On the official model, 135 = 15 × 9. The next cell checks that a global rotation does not enter those 135 numbers.


In [2]:
from official_mano import _pose_feature

full = np.concatenate([np.array([0.4, -0.2, 0.3]), np.zeros(45)])
feature = _pose_feature(full)
print("official pose-feature length", feature.size)
print("norm when only the wrist rotates", float(np.linalg.norm(feature)))
assert feature.size == 135
assert np.allclose(feature, 0.0)


official pose-feature length 135
norm when only the wrist rotates 0.0


## Check

You should be able to split 135 into 15 × 9. Changing only `global_orient` leaves the difference between `v_posed` and `v_shaped` at 0.

The next chapter combines `v_posed` with the new joint positions to make the final mesh.
